In [4]:
import numpy as np
import pandas as pd
import optuna
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Simulasi Dataset midterm-regresi-dataset.csv
np.random.seed(42)
n_samples = 3000
features = {f'audio_feature_{i}': np.random.randn(n_samples) for i in range(10)}
df_reg = pd.DataFrame(features)
# Target berupa Tahun Rilis Lagu (misal rentang 1990 - 2024)
df_reg['release_year'] = np.random.randint(1990, 2025, size=n_samples)

X_reg = df_reg.drop(columns=['release_year'])
y_reg = df_reg['release_year']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print("Data Preprocessing Soal 2 Selesai!")

Data Preprocessing Soal 2 Selesai!


In [5]:
# Training dan evaluasi menggunakan scikit-learn MLPRegressor (menghindari TensorFlow)
from sklearn.neural_network import MLPRegressor

def objective_reg(trial):
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    num_units = trial.suggest_categorical('num_units', [16, 32, 64])
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)

    with mlflow.start_run(nested=True):
        hidden_layers = (num_units,)
        clf = MLPRegressor(hidden_layer_sizes=hidden_layers, alpha=alpha, learning_rate_init=lr, max_iter=300, random_state=42)
        clf.fit(X_train_reg_scaled, y_train_reg)
        preds = clf.predict(X_test_reg_scaled)
        mse = mean_squared_error(y_test_reg, preds)
        mlflow.log_param("num_units", int(num_units))
        mlflow.log_param("alpha", float(alpha))
        mlflow.log_param("learning_rate", float(lr))
        mlflow.log_metric("mse", float(mse))

    return mse

# Jalankan Optuna dengan parent MLflow run untuk menghindari bentrok param
mlflow.set_experiment("UAS_Song_Regression")
with mlflow.start_run():
    study_reg = optuna.create_study(direction="minimize")
    study_reg.optimize(objective_reg, n_trials=5)

# Melatih Model Terbaik untuk mengambil nilai metrik lengkap
best_params = study_reg.best_params
best_lr = best_params.get('learning_rate')
best_units = int(best_params.get('num_units'))
best_alpha = best_params.get('alpha')
final_model = MLPRegressor(hidden_layer_sizes=(best_units,), alpha=best_alpha, learning_rate_init=best_lr, max_iter=300, random_state=42)
final_model.fit(X_train_reg_scaled, y_train_reg)

# Evaluasi
y_pred_reg = final_model.predict(X_test_reg_scaled)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n========== EVALUASI METRIK REGRESI ==========")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2 Score : {r2:.4f}")


[I 2026-06-23 22:39:48,854] A new study created in memory with name: no-name-867b1f7b-c1f2-4548-aaf0-4d1ff55c635e
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-06-23 22:39:49,655] Trial 0 finished with value: 102965.66549628148 and parameters: {'learning_rate': 0.002131720945258169, 'num_units': 32, 'alpha': 0.000280192657777619}. Best is trial 0 with value: 102965.66549628148.
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-06-23 22:39:50,201] Trial 1 finished with value: 1986784.1383066683 and parameters: {'learning_rate': 0.0011540689477417248, 'num_units': 16, 'alpha': 0.01741435100285918}. Best is trial 0 with value: 102965.6


========== EVALUASI METRIK REGRESI ==========
MSE  : 77184.3776
RMSE : 277.8208
MAE  : 223.8136
R2 Score : -760.9299


c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:
import lime
import lime.lime_tabular

# Menjelaskan bagaimana fitur audio mempengaruhi prediksi tahun rilis menggunakan LIME
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_reg_scaled),
    feature_names=list(X_reg.columns),
    mode='regression',
)

# Ambil 1 contoh baris data untuk diinterpretasikan
exp = explainer.explain_instance(X_test_reg_scaled[0], final_model.predict, num_features=5)
print("\n=== INTERPRETASI LIME UNTUK PREDIKSI BARIS PERTAMA ===")
print(exp.as_list())


=== INTERPRETASI LIME UNTUK PREDIKSI BARIS PERTAMA ===
[('-0.00 < audio_feature_6 <= 0.69', -83.50687993031988), ('-0.70 < audio_feature_2 <= -0.00', -78.10385565437673), ('-0.67 < audio_feature_0 <= 0.00', -78.02852740895628), ('-0.67 < audio_feature_5 <= 0.02', -75.5351198247555), ('-0.67 < audio_feature_1 <= 0.02', -74.47540237237261)]
